# Data Loading

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("ai-medical-chatbot.csv")

In [3]:
df.head()

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [4]:
df.shape

(256916, 3)

In [5]:
df.iloc[0]["Description"]

'Q. What does abutment of the nerve root mean?'

In [6]:
df.head()

,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [7]:
df["Document"] = df["Description"].fillna("")+ '\n\n' + df["Patient"].fillna("")+ '\n\n' + df["Doctor"].fillna("")

In [8]:
df["Document"].iloc[0]

'Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'

In [31]:
from dotenv import load_dotenv
import os
load_dotenv()


True

# Vector DB

In [10]:
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.docstore.document import Document

In [11]:
create_embedding = OpenAIEmbeddings(model="text-embedding-ada-002",api_key=os.environ["OPENAI_KEY"])

In [12]:
dir = os.environ["VECTORSTORE_DIRECTORY"]
if os.path.exists(dir) and os.listdir(dir):
    vectorstore = Chroma(embedding_function=create_embedding,persist_directory=dir)
else:
    documents = [Document(page_content=row["Document"]) for idx,row in df.head(100).iterrows()]
    
    vectorstore = Chroma.from_documents(documents=documents, persist_directory=dir, embedding=create_embedding)
    vectorstore.persist()

/var/folders/g7/d76pssrs7133khtcy6s2b_y80000gn/T/ipykernel_94214/1120768598.py:8: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [13]:
retriever = vectorstore.as_retriever()

# Prompt

In [14]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [15]:
# Define the system prompt that will guide the chatbot
system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise. If the question is not clear, ask follow-up questions."
    "\n\n"
    "{context}"
)

abhi tak hamne kya kiya pehle to hamde data load kiya using pandas then hamne preprocessing kari
phir hamne vector database initialize kiya and uske andar documents aur uski embeddings daali (also retrieval object baanya)
uske baad apan ab system prompt banayenge which will be made up of the instruction and the context depending on the query

In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human","{input}")
])

# Model

In [17]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o",temperature=0.2,api_key=os.environ["OPENAI_KEY"])

# RAG

In [18]:
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
retrieval_chain = create_retrieval_chain(retriever,document_chain)

In [30]:
query = "What does abutment of the nerve root mean?"
response = retrieval_chain.invoke({"input":query})
response

{'input': 'What does abutment of the nerve root mean?',
 'context': [Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'),
  Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'),
  Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'),
  Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->')],
 'answer': 'Abutment of the nerve root refers to a condition where a structure, such as a herniated disc or bone spur, is in close contact with the nerve root but not compressing it. It can cause symptoms like pain or discomfort due to irritation. For specific concerns, consulting a neurologist is recommended.'}

{'input': 'What does abutment of the nerve root mean?',
 'context': [Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'),
  Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'),
  Document(metadata={}, page_content='Q. What does abutment of the ne

# RAG (with memory)

In [20]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain.chains import ConversationalRetrievalChain

In [21]:
rag_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)
chat_history = []

In [22]:
question = "What does abutment of the nerve root mean?"
ai_msg_1 = rag_chain.invoke({
    "question":question,
     "chat_history":chat_history
})
chat_history.append((question,ai_msg_1["answer"]))
ai_msg_1["answer"]

'Abutment of the nerve root refers to a situation where a structure, such as a herniated disc or bone spur, is in close contact with a nerve root in the spine. This contact can potentially lead to irritation or compression of the nerve, which may cause symptoms such as pain, numbness, or weakness in the areas served by the affected nerve. For further information and personalized advice, it is recommended to consult a neurologist.'

In [23]:
question = "How to treat it?"
ai_msg_2 = rag_chain.invoke({
    "question":question,
     "chat_history":chat_history
})
chat_history.append((question,ai_msg_2["answer"]))


In [24]:
print(ai_msg_1["source_documents"])

[Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'), Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?\n\nHi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'), Document(metadata={}, page_content='Q. What does abutment of the nerve root mean?\n\nHi doctor,I am just wondering what is abutting and abu

In [25]:
ai_msg_2.keys()

dict_keys(['question', 'chat_history', 'answer', 'source_documents'])

# Agentic RAG

2 tools 
1) Medical retriever
2) Google search through "Tavily"

## Tool 1

In [32]:
from langchain.agents.agent_toolkits import create_retriever_tool
from langchain_openai import ChatOpenAI

In [33]:
retiever_tool = create_retriever_tool(retriever=retriever, name="Medical_diagnosis",description="Retrieve content for medical query")

## Tool 2

In [40]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily_tool = TavilySearchResults(tavily_api_key=os.environ["Tavily_API_KEY"])

In [ ]:
tavily_tool.run("Allahoakbar")

[{'title': 'Allahu Akbar (God is Greater) - WikiIslam',
  'url': 'https://wikiislam.net/wiki/Allahu_Akbar_(God_is_Greater)',
  'content': 'The phrase Allahu Akbar (الله أكبر) is a common phrase used by Muslims in various situations, including the Salah (obligatory five prayers a day) and has even been used in the past by some non-Muslims as a show of support or in reference to the common Abrahamic god. It is widely conflated with the Muslims who shout it whilst engaged in Jihad.The literal translation of this phrase from',
  'score': 0.8674071},
 {'title': 'Allahu Akbar | Dictionary of Islam',
  'url': 'https://islamreference.com/definitions/a/allahu-akbar/',
  'content': 'Allahu Akbar (الله أكبر) is an Arabic phrase meaning "God is greater." Known as the takbir (تَكْبِير), this utterance conveys the omnipotence and supreme authority of God over all aspects of existence. As such, it underscores the foundational monotheistic principle in Islam that nothing compares to the greatness of A

## Agent initialization

In [43]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

In [46]:
prompt = hub.pull("hwchase17/react")
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


/Users/baqirhusain/IK/ik/lib/python3.13/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [47]:
tools = [tavily_tool,retiever_tool]

In [48]:
re_act_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

In [49]:
agent_executor = AgentExecutor(agent=re_act_agent, verbose=True, max_iterations=5, handle_parsing_errors=True,tools=tools,)

In [54]:
agent_executor.invoke({"input":"Where were last elections in canada and Who won and by what margin and what policy they passed first?"})



> Entering new AgentExecutor chain...
To answer this question, I need to find information about the most recent elections in Canada, who won, the margin of victory, and the first policy they passed. I will use the search tool to gather this information.

Action: tavily_search_results_json
Action Input: "Canada last elections 2023 results winner margin first policy passed"[{'title': '2025 Canadian federal election - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/2025_Canadian_federal_election', 'content': 'The result was a reversal of polling trends lasting from mid-2023 to January 2025, which had led to projections of the Conservatives winning in a landslide.( Carney\'s replacement of Justin Trudeau as Leader of the Liberal Party played a key role in the turnaround. With his extensive experience as a central banker and his perceived competence, Carney was seen as better equipped to handle the trade war launched by the U.S. and other major economic issues.( Two sitting party leader

{'input': 'Where were last elections in canada and Who won and by what margin and what policy they passed first?',
 'output': 'In the 2023 Canadian federal elections, the Liberal Party, led by Mark Carney, won. Specific details about the margin of victory and the first policy passed are not available in the search results. For more detailed information, refer to official sources or reputable news outlets.'}

In [60]:
agent_executor.invoke({"input":"What are HIV symptoms?"})



> Entering new AgentExecutor chain...
HIV symptoms can vary depending on the stage of the infection. To provide a comprehensive answer, I will retrieve medical information on this topic. 

Action: Medical_diagnosis
Action Input: "HIV symptoms"Q. Are my symptoms due to HIV infection? I had a high-risk exposure 15 months ago.

Hi doctor, After 15 months of unprotected sex, I went for HIV test called Comb AIDS test but, the result came back negative for both HIV-1 and 2. Again after three months, I redid the test, but even this time, the result was negative. I was not satisfied, so I went for ELISA test, and again the results came back negative. I consulted two general physicians, who told me that I do not have HIV. But, I am still worried because of the symptoms that I have. I am a male, who weighs 62 kg, and I am 6 feet tall. The symptoms i'm having are:  severe weakness, ulcers in mouth and tongue, fatigue, pain in legs while sleeping, itching in the private part, occasional sore thr

{'input': 'What are HIV symptoms?',
 'output': "HIV symptoms can vary depending on the stage of the infection. In the early stage, known as acute HIV infection, symptoms can appear 2-4 weeks after exposure and may include flu-like symptoms such as fever, sore throat, fatigue, swollen lymph nodes, and rash. These symptoms are often mild and can be mistaken for a common cold or flu.\n\nAs the infection progresses to the chronic stage, symptoms may not be present for years, but the virus continues to multiply and damage the immune system. During this stage, individuals may experience persistent swollen lymph nodes, weight loss, fever, diarrhea, and cough.\n\nIn the advanced stage, known as AIDS (acquired immunodeficiency syndrome), the immune system is severely damaged, and individuals may experience more severe symptoms such as rapid weight loss, recurring fever, prolonged swelling of the lymph glands, diarrhea that lasts for more than a week, sores of the mouth, anus, or genitals, pneum

In [61]:
agent_executor.invoke({"input":"What are HIV symptoms? What is the latest experimental solutions being conducted?"})



> Entering new AgentExecutor chain...
To answer the question comprehensively, I need to first identify the symptoms of HIV and then look for the latest experimental solutions or treatments being conducted. I will start by retrieving information on HIV symptoms.

Action: Medical_diagnosis
Action Input: "HIV symptoms"Q. Are my symptoms due to HIV infection? I had a high-risk exposure 15 months ago.

Hi doctor, After 15 months of unprotected sex, I went for HIV test called Comb AIDS test but, the result came back negative for both HIV-1 and 2. Again after three months, I redid the test, but even this time, the result was negative. I was not satisfied, so I went for ELISA test, and again the results came back negative. I consulted two general physicians, who told me that I do not have HIV. But, I am still worried because of the symptoms that I have. I am a male, who weighs 62 kg, and I am 6 feet tall. The symptoms i'm having are:  severe weakness, ulcers in mouth and tongue, fatigue, pai

{'input': 'What are HIV symptoms? What is the latest experimental solutions being conducted?',
 'output': 'Common symptoms of HIV include fever, chills, rash, night sweats, muscle aches, sore throat, fatigue, swollen lymph nodes, and mouth ulcers. In 2023, significant advancements in HIV treatment include FDA-approved long-acting injectables, research in gene editing, therapeutic vaccines, and innovative therapies and prevention strategies. These developments offer hope for better control and potential cures for HIV.'}